# Project - Airline AI Assistant

We'll now bring together what we've learned to make an AI Customer Support assistant for an Airline

In [1]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [2]:
# Initialization

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4o-mini"
openai = OpenAI()

# As an alternative, if you'd like to use Ollama instead of OpenAI
# Check that Ollama is running for you locally (see week1/day2 exercise) then uncomment these next 2 lines
# MODEL = "llama3.2"
# openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')


OpenAI API Key exists and begins sk-proj-


In [3]:
system_message = "You are a helpful assistant for an Airline called FlightAI. "
system_message += "Give short, courteous answers, no more than 1 sentence. "
system_message += "Always be accurate. If you don't know the answer, say so."

In [4]:
# This function looks rather simpler than the one from my video, because we're taking advantage of the latest Gradio updates

def chat(message, history):
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7880
* To create a public link, set `share=True` in `launch()`.


## Tools

Tools are an incredibly powerful feature provided by the frontier LLMs.

With tools, you can write a function, and have the LLM call that function as part of its response.

Sounds almost spooky.. we're giving it the power to run code on our machine?

Well, kinda.

In [5]:
# Let's start by making a useful function

ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f"Tool get_ticket_price called for {destination_city}")
    city = destination_city.lower()
    return ticket_prices.get(city, "Unknown")

In [6]:
get_ticket_price("Berlin")

Tool get_ticket_price called for Berlin


'$499'

In [7]:
# There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city. Call this whenever you need to know the ticket price, for example when a customer asks 'How much is a ticket to this city'",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [ ]:
이 코드는 LLM이 사용할 “Tool(도구)” — 즉, 함수의 정의를 딕셔너리 형태로 기술한 예시입니다.
하나씩 아주 쉽게 풀어볼게요 👇

🧩 코드의 전체 목적

LLM이 외부 함수를 호출하려면
“이 함수가 무엇을 하는지, 어떤 인자를 받는지”를 정확히 설명하는 구조가 필요합니다.

👉 그래서 price_function 은 “티켓 가격을 가져오는 함수(get_ticket_price)”를
모델에게 알려주는 함수 정의서 (Function Schema) 입니다.

🧱 코드 구조 분석
price_function = {
    "name": "get_ticket_price",


name: 실제 툴(또는 함수)의 이름
→ LLM이 "이름으로" 함수를 호출할 수 있게 함
예: "function_call": {"name": "get_ticket_price", "arguments": {...}}

    "description": "Get the price of a return ticket to the destination city. Call this whenever you need to know the ticket price, for example when a customer asks 'How much is a ticket to this city'",


description:
이 함수가 어떤 역할을 하는지 자연어로 설명
→ LLM이 언제, 왜 이 함수를 써야 하는지 “이해”할 수 있게 함
(즉, 모델이 “티켓 가격 알려줘”라는 요청을 받으면 이 함수를 써야겠다고 판단)

    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}


이 부분은 함수의 입력값(파라미터) 구조를 JSON Schema 형태로 명시합니다.

🔍 세부 설명

"type": "object"
→ 함수의 인자는 객체(딕셔너리) 형태임을 의미
(예: {"destination_city": "Seoul"})

"properties"
→ 객체 내부에 어떤 키-값 쌍이 들어갈 수 있는지를 정의

"destination_city"
→ 이 함수는 여행 목적지 도시 이름을 문자열로 입력받음
예: "Seoul", "New York"

"required": ["destination_city"]
→ 반드시 있어야 하는 필수 인자

"additionalProperties": False
→ 정의되지 않은 다른 인자는 허용하지 않음
(즉, destination_city 외에 아무 것도 넣지 말라는 뜻)

🧠 요약하면
항목	설명
name	LLM이 호출할 함수의 이름
description	함수의 목적과 언제 사용하는지 설명
parameters	함수가 어떤 입력값을 필요로 하는지 정의
required	반드시 입력해야 하는 필드
additionalProperties	정의된 것 외의 값은 거부
⚙️ 실제 예시

이 schema를 LLM에 등록하면,
사용자가 “서울 왕복 티켓 얼마예요?” 라고 묻는 순간 모델은 이렇게 판단합니다 👇

{
  "function_call": {
    "name": "get_ticket_price",
    "arguments": {
      "destination_city": "Seoul"
    }
  }
}


→ 그러면 앱이 실제로 get_ticket_price("Seoul") 함수를 실행하고,
그 결과(예: “₩120,000”)를 모델에 다시 전달하여 응답을 완성합니다.

In [8]:
# And this is included in a list of tools:

tools = [{"type": "function", "function": price_function}]

## Getting OpenAI to use our Tool

There's some fiddly stuff to allow OpenAI "to call our tool"

What we actually do is give the LLM the opportunity to inform us that it wants us to run the tool.

Here's how the new chat function looks:

In [9]:
def chat(message, history):
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        response, city = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        response = openai.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [10]:
# We have to write that function handle_tool_call:

def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    arguments = json.loads(tool_call.function.arguments)
    city = arguments.get('destination_city')
    price = get_ticket_price(city)
    response = {
        "role": "tool",
        "content": json.dumps({"destination_city": city,"price": price}),
        "tool_call_id": tool_call.id
    }
    return response, city

In [11]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7881
* To create a public link, set `share=True` in `launch()`.


Tool get_ticket_price called for Paris
Tool get_ticket_price called for Tokyo


In [ ]:
이 코드는 LLM이 "툴을 실제로 호출하고 결과를 반영하는 전체 과정" 을 보여주는 예시입니다.
즉, LLM ↔ Tool(함수) ↔ LLM 응답 의 왕복 흐름을 코드로 구현한 거예요.
하나씩 구조적으로 정리해드릴게요 👇

🧩 전체 흐름 요약

tools 리스트로 사용할 함수(도구)를 등록

LLM이 “툴을 써야겠다” 판단 시 → tool_call 요청 생성

handle_tool_call()이 그 요청을 실제로 실행

결과를 다시 LLM에게 넘겨서 최종 답변 생성

1️⃣ Tools 리스트 정의
tools = [{"type": "function", "function": price_function}]


LLM에 전달할 툴 목록 (list)

각 항목은 {"type": "function", "function": ...} 형태로 작성

여기서는 앞서 정의한 price_function 하나만 등록함
→ 즉, LLM은 "get_ticket_price" 함수 하나만 쓸 수 있음

2️⃣ Chat 함수 (메인 로직)
def chat(message, history):
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]


대화 내역 구성

시스템 프롬프트(system_message)

지금까지의 대화 기록(history)

새 사용자 입력(message)

이 구조는 일반적인 ChatGPT API 포맷과 동일함

    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)


LLM에게 “이 대화 + 사용할 수 있는 도구 목록”을 전달

tools 파라미터 덕분에 LLM이 “이럴 땐 함수를 써야겠다” 고 판단 가능

✅ 도구 호출 감지
    if response.choices[0].finish_reason=="tool_calls":


finish_reason == "tool_calls" 인 경우,
→ 모델이 “함수 호출을 원한다”고 신호를 보낸 것임

즉, "How much is a ticket to Seoul?" 같은 질문을 받으면
모델이 "get_ticket_price" 호출 요청을 반환함

3️⃣ 실제 함수 호출 처리
        message = response.choices[0].message
        response, city = handle_tool_call(message)


모델이 반환한 메시지를 handle_tool_call() 로 전달

handle_tool_call() 함수가 실제 함수를 실행하고,
그 결과를 LLM에게 전달할 메시지를 만들어 반환

        messages.append(message)
        messages.append(response)
        response = openai.chat.completions.create(model=MODEL, messages=messages)


모델의 “함수 호출 요청 메시지” + “실행 결과 메시지”를 함께 다시 전달

이렇게 하면 LLM은 실행된 결과(예: "Seoul 티켓 가격은 ₩120,000입니다")를
자연스러운 문장으로 완성함

4️⃣ handle_tool_call() 함수 상세
def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    arguments = json.loads(tool_call.function.arguments)
    city = arguments.get('destination_city')


모델이 요청한 함수 호출 정보를 가져옴

어떤 함수(tool_call.function.name)

어떤 인자(arguments)

인자(JSON 문자열 형태)를 Python 객체로 변환 (json.loads)

여기선 destination_city 값을 꺼냄

    price = get_ticket_price(city)


실제 Python에서 정의된 함수 실행
→ 예: get_ticket_price("Seoul")
→ 결과: 120000 같은 숫자 반환

    response = {
        "role": "tool",
        "content": json.dumps({"destination_city": city,"price": price}),
        "tool_call_id": tool_call.id
    }


이건 “함수 실행 결과를 LLM에게 전달할 메시지”

role 은 "tool" — 즉, “이건 툴 실행 결과야” 라고 알려주는 역할

content 는 함수의 결과를 JSON으로 담음

tool_call_id 로 어떤 호출의 응답인지 연결

    return response, city


완성된 응답 메시지(response)와 도시명(city)을 반환

🧠 전체 호출 흐름 시각화
사용자 → chat()
   ↓
LLM (모델)
   ↓
(tool_calls 발생)
   ↓
handle_tool_call()
   ↓
get_ticket_price() 실행
   ↓
실행결과를 LLM에게 다시 전달
   ↓
LLM이 최종 자연어 응답 생성

🔧 예시 상황

사용자 입력:

“How much is a ticket to Seoul?”

1️⃣ 모델 판단: “price_function” 호출 필요
→ finish_reason="tool_calls"

2️⃣ handle_tool_call 실행
→ get_ticket_price("Seoul") = 120000

3️⃣ LLM 재호출
→ “A return ticket to Seoul costs ₩120,000.”

요약하자면 👇

이 코드는 “LLM이 도구를 스스로 선택하고, 호출하고, 결과를 받아 다시 대답하는”
LLM-Agent 구조의 기본 골격입니다.

In [ ]:
response.choices[0].finish_reason 은

“AI가 이번 대답을 어떤 이유로 멈췄는가”를 알려주는 표시입니다.

즉,
AI가 “지금 대답을 끝낸 이유(reason)”를 알려주는 신호예요.

⚙️ 일반적으로 나올 수 있는 finish_reason 값들
값	의미
"stop"	AI가 자기 말이 끝났다고 판단 (보통 자연스러운 끝)
"length"	토큰 제한 때문에 멈춤
"content_filter"	안전 필터에 걸림
"tool_calls"	🧩 AI가 도구를 써야 한다고 판단함
💬 예시로 보기
📍 1. 일반 대화일 때
response = openai.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": "안녕?"}]
)

print(response.choices[0].finish_reason)


→ 결과: "stop"
👉 AI가 그냥 “안녕!” 하고 답을 마친 상황이에요.

📍 2. 도구가 등록되어 있을 때

이제 tools=[...] 를 등록해서
AI가 외부 함수를 사용할 수 있게 해둡니다.

response = openai.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": "How much is a ticket to Seoul?"}],
    tools=tools  # get_ticket_price 등록되어 있음
)


이 경우 모델은 이렇게 생각합니다 👇

“이건 내가 직접 답할 수 없고, 등록된 함수(get_ticket_price)를 써야겠다.”

그래서 실제 답변 대신 이런 결과를 반환합니다 👇

{
  "choices": [
    {
      "finish_reason": "tool_calls",
      "message": {
        "tool_calls": [
          {
            "function": {
              "name": "get_ticket_price",
              "arguments": "{\"destination_city\": \"Seoul\"}"
            }
          }
        ]
      }
    }
  ]
}


✅ 이때 finish_reason 이 "tool_calls" 인 이유는:

“AI가 스스로 판단하기에, 이건 말로 끝낼 수 없고
함수를 호출해야 답을 완성할 수 있다”고 표시한 것.

🧩 즉, 코드에서의 의미
if response.choices[0].finish_reason == "tool_calls":


이 조건문은 “AI가 지금 도구를 호출하려고 한다면”을 감지하는 부분이에요.

그래서 그 아래 코드에서는 👇
실제로 그 도구를 실행(handle_tool_call)해서 결과를 구한 뒤,
AI에게 다시 전달하는 과정을 수행합니다.

🔁 쉽게 비유하면
상황	AI의 행동	finish_reason
“안녕?”	“안녕하세요!” 하고 말함	"stop"
“서울행 티켓 얼마야?”	“음, 그건 계산해봐야 하네. 도구를 써야겠다.”	"tool_calls"

즉 👇

finish_reason == "tool_calls" 는
AI가 ‘말로 끝내지 않고, 실제 함수를 불러야 한다’고 판단했음을 의미합니다.

원하신다면 실제 response 객체 구조(JSON 형태)도 예시로 보여드릴까요?
AI가 "tool_calls"을 반환할 때 내부 데이터가 어떻게 생겼는지 볼 수 있어요.

In [ ]:
💡 당신의 말 정리

GPT가 finish_reason = "tool_calls" 로 반환하는 이유는
우리가 tools=[...] 로 함수 구조를 JSON 형태로 미리 전달했기 때문이고,
그 정보가 모델의 입력 토큰 안에 포함되어 있어서
사용자의 질문이 그 함수 설명과 연관되면
“이건 도구를 써야겠다” 판단 후
대답을 멈추고 finish_reason="tool_calls" 로 끝낸다.

👉 ✅ 거의 완벽하게 맞습니다.
단, 내부적으로 조금 더 구체적인 순서가 있습니다.

🧩 전체 흐름 (조금 더 기술적으로)
① tools=[...] 전달 → “도구 설명도 프롬프트 일부가 됨”

당신이

tools = [{"type": "function", "function": price_function}]


이렇게 전달하면,
OpenAI API 내부에서는 이 내용이 시스템 프롬프트처럼 토큰화되어
모델 입력에 포함됩니다.

즉, 모델은 이런 “맥락”을 함께 읽습니다 👇

“내가 지금 사용할 수 있는 함수들이 어떤 이름을 가지고 있고,
어떤 인자(parameters)를 받아야 하는지”

② 사용자가 질문 → 모델이 내부적으로 reasoning (추론)

사용자가 “How much is a ticket to Seoul?” 이라고 물으면,
모델은 입력된 모든 토큰(= 대화 내용 + tools 정의)을 함께 보고 추론합니다.

그 결과:

“이건 내가 직접 말로 대답하기보다
등록된 함수(get_ticket_price)를 호출해야 한다”
라고 판단함.

③ 모델이 함수 호출을 “출력으로 생성”

모델이 실제로 내뱉는 출력 토큰 시퀀스가 이런 식으로 구성돼요 👇

{
  "tool_calls": [
    {
      "function": {
        "name": "get_ticket_price",
        "arguments": "{\"destination_city\": \"Seoul\"}"
      }
    }
  ]
}


즉, 모델은 **“함수를 호출하는 JSON 형식의 문장”**을 생성하고 멈춥니다.
(이게 곧 finish_reason = "tool_calls" 인 상태예요.)

④ finish_reason="tool_calls" 의 의미

이건 모델이 말로 “끝낸 게 아니라”

“도구 호출을 완성하고 멈췄다”
라는 상태를 명시적으로 알려주는 플래그입니다.

그래서 API는 이를 감지해서
“아, 모델이 지금 도구를 쓰려고 하는구나” 하고
message.tool_calls 속성에 함수 이름과 인자를 담아 돌려줍니다.

⚙️ 즉, 요약하면
단계	설명	내부에서 일어나는 일
1️⃣	tools 전달	함수의 구조와 설명이 LLM 입력 토큰에 포함
2️⃣	사용자가 질문	모델이 “이건 도구 써야겠다” 추론
3️⃣	모델 출력 생성	"tool_calls": [{"name": "...", "arguments": "..."}] JSON 생성
4️⃣	finish_reason 설정	"tool_calls" 로 종료 신호 전달
5️⃣	외부 코드 실행	실제 Python 함수 실행 후 결과를 다시 모델에 전달
🧠 핵심 문장으로 요약

✅ finish_reason="tool_calls" 는
LLM이 입력받은 tools 정의(=함수 스펙) 를 이해하고,
해당 함수 호출을 출력으로 생성했기 때문에 생기는 결과입니다.
즉, 모델이 “이건 내가 말할 게 아니라 도구를 실행해야 해”라고 판단한 신호예요.

In [ ]:
✅ 네, 모델이 매번 “finish_reason”을 출력하긴 하지만,
그건 모델이 직접 만든 게 아니라 API(시스템)가 모델의 출력 상태를 해석해서 붙이는 값이에요.

⚙️ 정리하자면
🧠 1️⃣ 모델은 "finish_reason"이라는 단어를 직접 출력하지 않아요

LLM(예: GPT-4)은 단순히 토큰 시퀀스만 생성합니다.
즉, 모델은 그저 “다음 토큰을 예측해서 계속 생성하다가” 멈출 뿐이에요.

예를 들어 👇
모델이 생성하다가 멈추는 이유는 다양하죠:

토큰 한도가 다 됐거나

“stop sequence”를 만났거나

내부적으로 도구 호출 JSON을 완성했다고 판단했거나

모델 자신은 “내가 왜 멈췄는지”를 별도로 말하지 않아요.
(그건 API가 판단합니다.)

🧩 2️⃣ finish_reason은 “API가 결과를 감싸며 추가한 메타데이터”

즉, 모델이 토큰 생성을 멈춘 시점에서
OpenAI API는 이렇게 판단합니다 👇

상황	API가 붙여주는 finish_reason
모델이 정상적으로 문장을 마쳤을 때	"stop"
토큰 제한(max_tokens)에 걸렸을 때	"length"
안전 필터에 걸렸을 때	"content_filter"
모델이 도구 호출을 완료했을 때	"tool_calls"

그래서 response.choices[0].finish_reason 은

“이 모델이 왜 멈췄는지를 시스템이 해석해 붙인 태그”예요.

🧪 예시
response = openai.chat.completions.create(
    model="gpt-4o",
    messages=[{"role":"user", "content":"Hello"}]
)


👉 모델은 “Hello!” 라는 문장만 출력했지만
API는 그 결과를 이런 식으로 감싸서 반환합니다 👇

{
  "choices": [
    {
      "message": {"role": "assistant", "content": "Hello!"},
      "finish_reason": "stop"
    }
  ]
}

⚙️ 3️⃣ Tool 사용 시엔 "tool_calls" 감지 로직 추가

만약 tools=[...] 가 포함된 상태라면,
모델이 JSON 형태의 함수 호출을 출력하는 순간
API는 "finish_reason": "tool_calls" 로 인식해서 알려줍니다.

📦 4️⃣ 즉, 정리하면
구분	누가 생성하나	의미
message.content	🧠 모델이 직접 생성	실제 텍스트 / 함수 호출 JSON
finish_reason	⚙️ API가 자동 추가	모델이 “왜 멈췄는가”를 알려주는 태그
✅ 결론

모델이 항상 finish_reason을 “출력”하는 건 아니고,
항상 finish_reason이 “결과로 포함되어 반환되긴 한다.
(단, 이건 모델이 만든 게 아니라 OpenAI API가 붙여주는 메타데이터입니다.)

In [ ]:
맞아요. 그리고 여기서 “finish_reason이 tool_calls일 때 특별한 일이 벌어집니다.”

즉, 단순히 "finish_reason": "tool_calls" 라는 태그만 주는 게 아니라,
👉 함수 호출에 대한 구체적인 정보(tool_calls 필드) 까지 같이 JSON 구조로 반환됩니다.

아래에서 아주 쉽게 정리해드릴게요 👇

🧩 1️⃣ 일반적인 경우 (finish_reason = "stop")

AI가 단순히 텍스트 답변만 했을 경우 👇

{
  "choices": [
    {
      "finish_reason": "stop",
      "message": {
        "role": "assistant",
        "content": "A return ticket to Seoul costs ₩120,000."
      }
    }
  ]
}


➡️ 즉, 모델은 말로 답변을 끝냈기 때문에
message.content 안에 자연어 텍스트가 들어 있고,
finish_reason 은 "stop" 으로 표시됩니다.

🧠 2️⃣ 도구 호출인 경우 (finish_reason = "tool_calls")

이건 완전히 달라요.
모델이 “지금은 말로 답변할 수 없고, 등록된 함수(tool)를 호출해야겠다”
라고 판단한 경우입니다.

이때의 응답 예시는 다음과 같아요 👇

{
  "choices": [
    {
      "finish_reason": "tool_calls",
      "message": {
        "role": "assistant",
        "tool_calls": [
          {
            "id": "call_12345",
            "type": "function",
            "function": {
              "name": "get_ticket_price",
              "arguments": "{\"destination_city\": \"Seoul\"}"
            }
          }
        ]
      }
    }
  ]
}

✅ 여기서 중요한 점

"finish_reason": "tool_calls"
→ “모델이 함수를 호출해야 한다”고 알려주는 신호

"message.tool_calls"
→ “그 함수가 무엇이고, 어떤 인자를 줘야 하는지” 구체적인 JSON 정보

즉, API는 이 두 가지를 한 세트로 반환합니다:

① finish_reason 으로 ‘지금은 도구 호출 상태’임을 표시
② message.tool_calls 로 실제 호출할 함수와 인자를 제공

⚙️ 3️⃣ 그래서 코드에서는 이렇게 씁니다
if response.choices[0].finish_reason == "tool_calls":
    message = response.choices[0].message
    tool_call = message.tool_calls[0]
    arguments = json.loads(tool_call.function.arguments)


finish_reason 이 "tool_calls" 일 때만
→ message.tool_calls 에 접근 가능

그 안의 arguments (문자열 JSON)를 파싱해서
실제 Python 함수에 넣을 수 있게 변환

🧩 4️⃣ 전체 구조 정리
항목	위치	설명
finish_reason	response.choices[0].finish_reason	모델이 “왜 멈췄는가”
message.role	"assistant"	모델의 발화자 역할
message.content	없음 (이 경우)	자연어 답변이 아니라 함수 호출임
message.tool_calls	JSON 배열	어떤 함수와 인자를 호출해야 하는지 구체적 설명
📊 한 줄 요약

✅ finish_reason == "tool_calls" 인 경우,
OpenAI API는 단순히 "tool_calls" 라고만 표시하지 않고,
어떤 함수(name), 어떤 인자(arguments)를 호출해야 하는지까지
함께 message.tool_calls JSON 필드로 반환합니다.

In [ ]:
이 부분은 LLM의 “도구 호출 흐름”을 완전히 이해하는 데 핵심이에요.

🧩 먼저 결론부터 말하면
message = response.choices[0].message


여기서 message 는

모델이 방금 생성한 "assistant 역할의 메시지" 전체 객체예요.

즉, 이건 단순한 문자열이 아니라
📦 딕셔너리(dict) 형태의 데이터입니다.
(내용은 자연어 문장일 수도 있고, 도구 호출일 수도 있음)

⚙️ 1️⃣ 일반적인 경우 (도구를 안 쓸 때)

모델이 그냥 “말로” 답했을 경우,
message 는 이런 구조예요 👇

{
  "role": "assistant",
  "content": "A return ticket to Seoul costs ₩120,000."
}


즉,

role: 메시지를 보낸 주체 ("assistant")

content: 실제 답변 텍스트

이건 우리가 평소에 보던 ChatGPT 응답이랑 똑같죠.

⚙️ 2️⃣ 도구 호출일 때 (finish_reason == "tool_calls")

이때는 말 대신 함수 호출 정보가 담긴 JSON 구조가 들어옵니다 👇

{
  "role": "assistant",
  "tool_calls": [
    {
      "id": "call_12345",
      "type": "function",
      "function": {
        "name": "get_ticket_price",
        "arguments": "{\"destination_city\": \"Seoul\"}"
      }
    }
  ]
}

✅ 각 필드 의미
키	의미
role	여전히 "assistant" — 모델이 도구 호출을 지시한 것
tool_calls	도구 호출 요청 리스트 (여러 개일 수도 있음)
tool_calls[0].function.name	호출할 함수 이름
tool_calls[0].function.arguments	함수 인자 (JSON 문자열 형태)
🔧 3️⃣ 코드 상에서 하는 일
message = response.choices[0].message


이 한 줄로,
모델이 방금 “도구를 호출하라”고 응답한 이 전체 메시지를 가져옵니다.

그 다음 코드에서는 👇

response, city = handle_tool_call(message)


이 메시지를 handle_tool_call() 함수에 넘겨서
실제 Python 함수(get_ticket_price)를 실행하도록 하는 거예요.

즉, message 안에 들어 있는 정보는 이런 식으로 쓰이는 거죠:

message.tool_calls[0].function.name → 어떤 함수 부를지

message.tool_calls[0].function.arguments → 인자(JSON) 파싱해서 사용

🧠 요약하자면
상황	message의 형태	내용
일반 텍스트 답변	{"role": "assistant", "content": "문장"}	자연어
도구 호출 발생	{"role": "assistant", "tool_calls": [...]}	함수 이름과 인자 정보
💬 예시로 보면 이렇게 작동합니다

사용자:

“How much is a ticket to Seoul?”

모델 응답(message) 👇

{
  "role": "assistant",
  "tool_calls": [
    {
      "function": {
        "name": "get_ticket_price",
        "arguments": "{\"destination_city\": \"Seoul\"}"
      }
    }
  ]
}


이걸 handle_tool_call() 이 받아서
👉 get_ticket_price("Seoul") 실행
👉 그 결과를 LLM에게 다시 전달

요약 한 줄로 말하면 👇

response.choices[0].message 는
“모델이 방금 생성한 assistant 메시지 전체 객체”이며,
이 안에는 자연어 텍스트(content) 또는 도구 호출 정보(tool_calls) 가 들어 있습니다.

In [ ]:
이 부분이 바로 “모델이 요청한 함수 호출을 실제로 실행하고, 그 결과를 다시 모델에게 돌려주는 핵심 로직”이에요.
하나씩 쉽게, 흐름 중심으로 설명해볼게요 👇

🧠 전체 역할 요약

handle_tool_call() 은 이름 그대로

“AI가 요청한 tool(function)을 실제로 실행하고, 결과를 포장해서 다시 돌려주는 함수”
입니다.

⚙️ 동작 순서 한눈에 보기

1️⃣ 모델이 함수 호출을 요청함 → message.tool_calls 에 저장됨
2️⃣ 이 함수가 그 내용을 꺼내서 → 실제 Python 함수 실행
3️⃣ 결과를 JSON 형태로 만들어서 → 모델에게 “이게 실행 결과야!” 라고 전달

🔍 한 줄씩 상세히 해석하기
① 함수 시작
def handle_tool_call(message):


매개변수 message 는 바로 앞 단계의
response.choices[0].message
(즉, 모델이 보낸 “tool 호출 요청 메시지”)입니다.

② 어떤 도구를 호출하려는지 꺼내기
tool_call = message.tool_calls[0]


모델이 한 번에 여러 도구를 호출할 수도 있어서 리스트 형태입니다.

여기서는 첫 번째 도구 호출 정보(tool_calls[0])를 가져와요.

👉 예를 들어 tool_call 안에는 이런 정보가 들어 있습니다:

{
  "id": "call_123",
  "type": "function",
  "function": {
    "name": "get_ticket_price",
    "arguments": "{\"destination_city\": \"Seoul\"}"
  }
}

③ 인자(arguments) 꺼내기
arguments = json.loads(tool_call.function.arguments)


tool_call.function.arguments 는 문자열 형태의 JSON이에요.
(예: '{"destination_city": "Seoul"}')

json.loads() 로 파싱하면 파이썬 딕셔너리로 변환됩니다:

{"destination_city": "Seoul"}

④ 도시 이름 꺼내기
city = arguments.get('destination_city')


딕셔너리에서 'destination_city' 키를 꺼내옵니다.
👉 city = "Seoul"

⑤ 실제 Python 함수 실행
price = get_ticket_price(city)


이제 진짜로 우리가 정의한 Python 함수 실행!
(즉, LLM이 요청한 툴을 “실제로 실행하는 단계”)

get_ticket_price("Seoul") 이 실행되어,
예를 들어 price = 120000 을 반환한다고 해볼게요.

⑥ 모델에게 결과를 포장해서 돌려주기
response = {
    "role": "tool",
    "content": json.dumps({"destination_city": city,"price": price}),
    "tool_call_id": tool_call.id
}


이게 아주 중요해요 ⚠️
이 부분이 바로 **“실행 결과를 모델에게 다시 돌려주는 형식”**이에요.

필드	설명
"role": "tool"	이건 사람이 한 말이 아니라 “도구의 실행 결과다” 라는 표시
"content"	함수 실행 결과를 JSON 문자열로 전달
"tool_call_id"	어떤 호출 요청에 대한 결과인지 연결하기 위한 ID (모델이 이걸로 매칭함)

👉 최종적으로 이렇게 생긴 딕셔너리를 만듭니다:

{
  "role": "tool",
  "content": "{\"destination_city\": \"Seoul\", \"price\": 120000}",
  "tool_call_id": "call_123"
}

⑦ 함수 종료
return response, city


response: 모델에게 전달할 결과 메시지

city: 나중에 로그나 출력용으로 따로 사용할 수 있게 반환

🔁 전체 흐름 요약

1️⃣ 모델:

“get_ticket_price(‘Seoul’) 호출할게!”

↓

2️⃣ handle_tool_call():

모델의 호출 요청(message.tool_calls[0])을 읽음

실제 함수 실행

결과를 role='tool' 메시지로 포장해서 반환

↓

3️⃣ 이 결과를 다시 모델에게 전달
→ 모델은 최종 문장으로 응답

“A return ticket to Seoul costs ₩120,000.”

🧠 한 문장으로 요약하면

handle_tool_call() 은
“AI가 요청한 함수 호출을 실제 코드로 실행하고,
결과를 LLM이 이해할 수 있는 JSON 메시지로 돌려주는 브릿지 역할”